# Model training

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor 

from scipy import stats

from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, roc_curve, RocCurveDisplay, classification_report, ConfusionMatrixDisplay, PrecisionRecallDisplay

import joblib

# needed for the pipeline transformer
import sys
sys.path.append(str(Path().cwd().parents[0] / "src"))
from components.transformations import calculate_deviations

In [ ]:
data_location = Path().cwd().parent / "src" / "artifacts" / "data"
data_location

In [ ]:
df = pd.read_csv(data_location / "baseline_data.csv")
df

In [ ]:
# creating the baseline dataset
X = df.drop(["id", "diagnosis"], axis=1)
y_raw = df.pop("diagnosis")

In [ ]:
y_raw.unique()

In [ ]:
# splitting the data
# raw = uses labels as B and M
X_train, X_test, y_train_raw, y_test_raw = train_test_split(X, y_raw, test_size=0.2, stratify=y_raw, shuffle=True, random_state=42)

In [ ]:
fig, axs = plt.subplots(figsize=(12,6), nrows=1, ncols=2)
sns.countplot(x=y_train_raw.sort_values(), stat="percent", hue=y_train_raw.sort_values(), ax=axs[0])
axs[0].set_title("Training data")
sns.countplot(x=y_test_raw.sort_values(),  stat="percent", hue=y_test_raw.sort_values(),  ax=axs[1])
axs[1].set_title("Test data")
for ax in axs:
    ax.spines[["top", "right", "bottom", "left"]].set_visible(False)
    ax.grid(True, linewidth=0.5, linestyle=':', axis="y")

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
# encode labels
label_encoder = LabelEncoder()
y_train = pd.Series(label_encoder.fit_transform(y_train_raw), index=y_train_raw.index, name="y_train")
y_test = pd.Series(label_encoder.transform(y_test_raw), index=y_test_raw.index, name="y_test")

In [ ]:
# labels
labels = dict(
    zip(
        label_encoder.classes_,
        label_encoder.transform(label_encoder.classes_)
    )
)
labels

# Baselines

The following models are investigated as baselines for the task.  
Keeping all features in the dataset, what are the most promising models regarding performance and interpretability?

In [ ]:
def run_pipeline(X, y, model, return_scores=False):

    # cross validation strategy
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    
    model_name, model_type = model
    
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        (model_name, model_type)
    ])

    if hasattr(model_type, "predict_proba"):
        method = "predict_proba"
        cv_scores = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1, verbose=3)
        y_pred = cv_scores.argmax(axis=1)

        if return_scores:
            return cv_scores[:,1], y_pred

        return {
        "model"     : model_name,
        "method"    : method,
        "precision" : precision_score(y_pred=y_pred, y_true=y),
        "recall"    : recall_score(y_pred=y_pred, y_true=y),
        "f1_score"  : f1_score(y_pred=y_pred, y_true=y),
        "roc_auc"   : roc_auc_score(y_true=y, y_score=cv_scores[:,1])
    }
        
    else:
        # All methods investigatated have a predict_proba method
        # in case a new model with a decision_function method is added
        # This must be updated
        method = "decision_function"
        cv_scores = cross_val_predict(pipe, X, y, cv=cv, method="decision_function", n_jobs=-1, verbose=3)

        return {
        "model"     : model_name,
        "method"    : method,
        "precision" : np.nan,
        "recall"    : np.nan,
        "f1_score"  : np.nan,
        "roc_auc"   : np.nan
    }

    

In [ ]:
# Baseline models with some defalut parameters
models = [
    ("logistic_regression", LogisticRegression(solver="liblinear", l1_ratio=1, C=1, max_iter=1000)),
    ("knn", KNeighborsClassifier(n_neighbors=5, metric="euclidean")),
    ("decision_tree", DecisionTreeClassifier(criterion="entropy", max_depth=4, splitter="best")),
    ("forest", RandomForestClassifier(n_estimators=100, criterion="entropy")),
    ("gradient_boosting", GradientBoostingClassifier()),
    ("adaboost", AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=4), n_estimators=100)),
    ("support_vector", CalibratedClassifierCV(SVC(C=0.5, kernel="rbf", gamma="auto"), ensemble=False)), # avoid sklearn warning
    ("gaussian_nb", GaussianNB()),
    ("mlp", MLPClassifier(hidden_layer_sizes=(50,50), activation="relu", early_stopping=True, batch_size=64))
]

In [ ]:
scores_baseline = []
for model in models:
    score = run_pipeline(X_train, y_train, model)
    scores_baseline.append(score)

## Baseline metrics

In [ ]:
# Using all features, Logistic Regression, support vector machines and KNN seem to be the most promesing models
# They already achieve high performance but interprertability is difficult
metrics_baseline = pd.DataFrame(scores_baseline).sort_values("f1_score", ascending=False, ignore_index=True)
metrics_baseline

# Feature selection

## Welch's test (ANOVA)
Since we are dealing with only two features, we can use a Welch's test instead of ANOVA to see whether there are any differences between the benign and maligant tumors for each feature.

In [ ]:
# merge the data again to use the label for grouping
X_for_feature_selection = pd.merge(X_train, y_train, left_index=True, right_index=True)
X_for_feature_selection.head()

In [ ]:
# separte into two groups B and M
benign    = X_for_feature_selection[X_for_feature_selection["y_train"] == 0]
malignant = X_for_feature_selection[X_for_feature_selection["y_train"] == 1] # This is the positive class what we want to detect

In [ ]:
display(benign.head())
display(malignant.head())

In [ ]:
def run_ttests(df1, df2):
    _, n_features = df1.shape
    results = {"feature" : [], "t_statistic" : [], "p_value_t" : []}
    for feature in df1.columns:
        # skip the label
        if feature == "y_train":
            continue
        # perform t-tests, covariance matrix shows the variances for both groups are different
        # so we use unequal variances
        t_statistic, p_value_t = stats.ttest_ind(df1[feature], df2[feature], equal_var=False, alternative="two-sided")
        results["feature"].append(feature)
        results["t_statistic"].append(t_statistic)
        results["p_value_t"].append(p_value_t)
        
    df = pd.DataFrame(results).sort_values(by=["p_value_t"], ascending=True, ignore_index=True)
    # adjust p-values using benjamini-hochberg
    df["p_value_bh_adjusted"] = stats.false_discovery_control(df["p_value_t"])
    # run bonferri comparison
    df["comparison_bonferri"] = df["p_value_t"].lt(0.05/n_features)
    # run bh comparison to see which hypothesis can be rejected
    df["comparison_bh"]       = df["p_value_bh_adjusted"].lt(0.05)

    return df

In [ ]:
df_ttest = run_ttests(benign, malignant)
df_ttest

In [ ]:
features_to_remove = df_ttest[(~df_ttest["comparison_bonferri"]) & (~df_ttest["comparison_bh"])].loc[:, "feature"].tolist()
features_to_remove

In [ ]:
X_train_after_fs = X_train.drop(features_to_remove, axis=1)
X_train_after_fs.shape

## Variance Inflation Factor

In [ ]:
def get_vif(X):
    features_column = []
    vif_column = []
    for idx, feature in enumerate(X.columns):
        features_column.append(feature)
        vif_column.append(variance_inflation_factor(X, idx))
    
    df_vif = pd.DataFrame(
        {
            "features" : features_column,
            "vif" : vif_column
        }
    )
    return df_vif.sort_values(by="vif", ascending=False, ignore_index=True)

In [ ]:
print(get_vif(X_train_after_fs))

## Model candidates
What is the performance of the candidate models after removing the features?

In [ ]:
scores_after_fs = []
for model in models:
    score = run_pipeline(X_train_after_fs, y_train, model)
    scores_after_fs.append(score)

## Feature selection metrics

In [ ]:
metrics_after_fs = pd.DataFrame(scores_after_fs).sort_values("f1_score", ascending=False, ignore_index=True)
metrics_after_fs

# Feature Engineering
Some of the features share a geometrics interpretation we can use to derive possible usefull features

In [ ]:
X_train_after_fs.columns.sort_values().tolist()

In [ ]:
X_train_reduced = X_train_after_fs.drop(["perimeter_mean", "perimeter_worst", "perimeter_se", "radius_mean", "radius_worst", "radius_se"], axis=1)
X_train_reduced.shape

In [ ]:
#X_train_reduced = X_train_after_fs.drop(["area_mean", "area_worst", "perimeter_mean", "perimeter_worst", "radius_mean", "radius_worst"], axis=1)
#X_train_reduced.shape

In [ ]:
X_train_reduced.columns.sort_values()

In [ ]:
for feature in X_train_reduced.columns:
    # ignore SE for the moment
    if feature.endswith("_se"):
        continue
    feature_name_template = feature.replace("_mean", "")
    feature_worst = f"{feature_name_template}_worst"
    # Both mean and worstmust be in the columns
    if feature_worst not in X_train_reduced.columns:
        continue
    
    if X_train_reduced[feature].eq(0).any():
        continue
        #X_train_reduced[f"{feature_name_template}_deaviation_abs"] = X_train_reduced.apply(
        #    lambda row: (row[f"{feature_name_template}_worst"] - row[f"{feature_name_template}_mean"]), axis=1
        #)
    else:
        X_train_reduced[f"{feature_name_template}_deviation_relative"] = X_train_reduced.apply(
            lambda row: (row[f"{feature_name_template}_worst"] - row[f"{feature_name_template}_mean"])/row[f"{feature_name_template}_mean"], axis=1
        )
    X_train_reduced = X_train_reduced.drop([feature_worst], axis=1)
X_train_reduced.shape

In [ ]:
X_train_reduced.columns.sort_values()

In [ ]:
X_train_engineered = X_train_reduced.drop(
    [
        'fractal_dimension_worst',
        'concave_points_mean',
        'concave_points_se',
        'concave_points_worst',
        'concavity_mean',
        'concavity_se',
        'concavity_worst',
    ], axis=1
)


In [ ]:
X_train_engineered.shape

In [ ]:
X_train_engineered.columns.sort_values().to_list()

In [ ]:
benign_and_malignant = pd.merge(X_train_engineered, y_train, left_index=True, right_index=True)
benign    = benign_and_malignant[benign_and_malignant["y_train"] == 0]
malignant = benign_and_malignant[benign_and_malignant["y_train"] == 1]
display(benign.head())
display(malignant.head())

In [ ]:
run_ttests(benign, malignant)

In [ ]:
get_vif(X_train_engineered)

In [ ]:
scores_engineered = []
for model in models:
    score = run_pipeline(X_train_engineered, y_train, model)
    scores_engineered.append(score)

## Feature Engineering metrics

In [ ]:
metrics_engineered = pd.DataFrame(scores_engineered).sort_values("f1_score", ascending=False, ignore_index=True)
metrics_engineered

## Feature Engineering Pipeline

In [ ]:
# SKIP THIS
# Once all decision about what features to keep are made we need a deterministc pipeline for all datasets
def calculate_deviations(df:pd.DataFrame) -> pd.DataFrame:

    features = ["area", "compactness", "smoothness", "symmetry", "texture"]
    features_to_drop = ["perimeter_mean", "perimeter_worst", "perimeter_se", "radius_mean", "radius_worst", "radius_se"]

    X = df.drop(features_to_drop, axis=1)

    transformed = {}
    for feature in features:
        transformed[f"{feature}_mean"] = X.loc[:, f"{feature}_mean"]
        transformed[f"{feature}_deviation_relative"] = (X.loc[:, f"{feature}_worst"] - X.loc[:, f"{feature}_mean"]) / X.loc[:, f"{feature}_mean"]

    for feature in features[:2]:
        transformed.update(
            {
                f"{feature}_se" : X.loc[:, f"{feature}_se"]
            }
        )

    return pd.DataFrame(transformed, index=X.index)

In [ ]:
transformer = FunctionTransformer(calculate_deviations)

In [ ]:
# The pipeline must take the complete dataset and perform the feature selection and engineering
X_train_transformed = transformer.transform(X_train)

In [ ]:
X_train_transformed.shape

In [ ]:
X_train_transformed.columns.sort_values()

In [ ]:
X_train_engineered.columns.sort_values()

In [ ]:
X_train_transformed.columns.sort_values() == X_train_engineered.columns.sort_values()

# Evaluating best models

In [ ]:
logistic_reg = [model for model in models if model[0]=="logistic_regression"][0]
logistic_scores, y_pred_logistc = run_pipeline(X_train_engineered, y_train, logistic_reg, return_scores=True)

svm = [model for model in models if model[0]=="support_vector"][0]
svm_scores, y_pred_svm = run_pipeline(X_train_engineered, y_train, svm, return_scores=True)

forest = [model for model in models if model[0]=="forest"][0]
forest_scores, y_pred_forest = run_pipeline(X_train_engineered, y_train, forest, return_scores=True)

In [ ]:
fig, axs = plt.subplots(figsize=(16,4), nrows=1, ncols=3, sharex=True, sharey=True)

RocCurveDisplay.from_predictions(y_train, logistic_scores, ax=axs[0], despine=True, name="LogisticRegression")
RocCurveDisplay.from_predictions(y_train, svm_scores,      ax=axs[1], despine=True, name="SVM")
RocCurveDisplay.from_predictions(y_train, forest_scores,   ax=axs[2], despine=True, name="RandomForest")

for ax in axs:
    ax.grid(True, linewidth=0.5, linestyle=":")
fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(figsize=(16,4), nrows=1, ncols=3, sharex=True, sharey=True)

PrecisionRecallDisplay.from_predictions(y_train, logistic_scores, ax=axs[0], despine=True, name="LogisticRegression")
PrecisionRecallDisplay.from_predictions(y_train, svm_scores,      ax=axs[1], despine=True, name="SVM")
PrecisionRecallDisplay.from_predictions(y_train, forest_scores,   ax=axs[2], despine=True, name="RandomForest")

for ax in axs:
    ax.grid(True, linewidth=0.5, linestyle=":")
fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(figsize=(16,4), nrows=1, ncols=3, sharex=True, sharey=False)

ConfusionMatrixDisplay.from_predictions(y_train, y_pred_logistc, ax=axs[0], normalize="true")
ConfusionMatrixDisplay.from_predictions(y_train, y_pred_svm,     ax=axs[1], normalize="true")
ConfusionMatrixDisplay.from_predictions(y_train, y_pred_forest,  ax=axs[2], normalize="true")

for ax, title in zip(axs, ("LogisticRegression", "SVM", "RandomForest")):
    ax.set_title(title)

In [ ]:
# Logistic Regression
report = classification_report(y_true=y_train, y_pred=y_pred_logistc)
print(report)

In [ ]:
# SVM
report = classification_report(y_true=y_train, y_pred=y_pred_svm)
print(report)

In [ ]:
# Forest
report = classification_report(y_true=y_train, y_pred=y_pred_forest)
print(report)

# Hyperparamater tuning

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True)

## Logistic Regression

In [ ]:
logistic_reg

In [ ]:
logistic_param_distributions = {
    "logistic_regression__C" : np.logspace(-3, 3, 100),
    "logistic_regression__l1_ratio" : np.linspace(0, 1, 51),
    "logistic_regression__solver" : ["saga"],
    "logistic_regression__max_iter" : [10000, 12000]
}

In [ ]:
logistic_pipe = Pipeline([
        ("transformer", transformer),
        ("scaler", StandardScaler()),
        logistic_reg
    ])
logistic_pipe

In [ ]:
logistic_search = RandomizedSearchCV(
    logistic_pipe,
    param_distributions=logistic_param_distributions,
    n_iter=1200,
    cv=cv,
    scoring="f1",
    error_score="raise"
)
logistic_search

In [ ]:
logistic_search.fit(X_train, y_train)

In [ ]:
logistic_search.best_score_

In [ ]:
logistic_best_scores = pd.DataFrame(logistic_search.cv_results_).sort_values(by="mean_test_score", ascending=False)
logistic_best_scores

In [ ]:
logistic_params = logistic_best_scores["params"][logistic_best_scores["mean_test_score"] >= 0.95].tolist()
logistic_params = pd.DataFrame(logistic_params)
logistic_params

In [ ]:
artifacts_dir = Path().cwd().parent / "src" / "artifacts" / "models"
artifacts_dir.exists()

In [ ]:
joblib.dump(logistic_search.best_estimator_, artifacts_dir.joinpath("logistic_regression.joblib"))

In [ ]:
best_model = logistic_search.best_estimator_

In [ ]:
y_pred_test = best_model.predict(X_test)
y_pred_scores_test = best_model.predict_proba(X_test)[:,1]

In [ ]:
# on test data
print(
    classification_report(y_pred=y_pred_test, y_true=y_test)
)

In [ ]:
fig, axs = plt.subplots(figsize=(12,4), nrows=1, ncols=2)

PrecisionRecallDisplay.from_predictions(y_true=y_test, y_score=y_pred_scores_test, ax=axs[0], despine=True)
RocCurveDisplay.from_predictions(y_true=y_test, y_score=y_pred_scores_test, ax=axs[1], despine=True)

for ax in axs:
    ax.grid(True, linewidth=0.5, linestyle=":")
fig.tight_layout()

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_test, normalize="true")

## Random Forest

In [ ]:
forest

In [ ]:
forest_param_distributions = {
    "forest__n_estimators" : np.arange(10, 1000, 10),
    "forest__criterion" : ["gini", "entropy"],
    "forest__max_depth" : np.arange(2,22,2),
    "forest__max_features" : [10, "sqrt", None]
}

In [ ]:
forest_pipe = Pipeline([
    ("transformer", transformer),
    ("scaler", StandardScaler()),
    forest
])

In [ ]:
forest_search = RandomizedSearchCV(
    forest_pipe,
    param_distributions=forest_param_distributions,
    n_iter=1200,
    cv=cv,
    scoring="f1",
    error_score="raise"
)
forest_search

In [ ]:
forest_search.fit(X_train, y_train)

In [ ]:
forest_search.best_score_

In [ ]:
forest_best_scores = pd.DataFrame(forest_search.cv_results_).sort_values(by="mean_test_score", ascending=False)
forest_best_scores

In [ ]:
forest_params = forest_best_scores["params"][forest_best_scores["mean_test_score"] >= 0.95].tolist()
forest_params = pd.DataFrame(forest_params)
forest_params

In [ ]:
joblib.dump(forest_search.best_estimator_, artifacts_dir.joinpath("forest.joblib"))

In [ ]:
best_forest = forest_search.best_estimator_

In [ ]:
y_pred_forest_test = best_forest.predict(X_test)
y_pred_forest_scores_test = best_forest.predict_proba(X_test)[:,1]

In [ ]:
# on test data
print(
    classification_report(y_pred=y_pred_forest_test, y_true=y_test)
)

In [ ]:
fig, axs = plt.subplots(figsize=(12,4), nrows=1, ncols=2)

PrecisionRecallDisplay.from_predictions(y_true=y_test, y_score=y_pred_forest_scores_test, ax=axs[0], despine=True)
RocCurveDisplay.from_predictions(y_true=y_test, y_score=y_pred_forest_scores_test, ax=axs[1], despine=True)

for ax in axs:
    ax.grid(True, linewidth=0.5, linestyle=":")
fig.tight_layout()

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_forest_test, normalize="true")